In [ ]:
"""
03_ref_biaised_integration.py  [optimized]
-------------------------------------------
Loads the classified patient file to derive dominant bias positions,
then injects each bias into a reference address dataset and geocodes
the variants in chunks.

Usage:
    python 03_ref_biaised_integration.py <*_clean_classified.csv> [OPTIONS]

Options:
    --gpu             CUDA device ID for the position analysis step (default: 3)
    --biaises-file    Path to biaises_identified.csv
    --reference-file  Reference address CSV (auto-detected if omitted)
    --sample-size     Rows to sample from reference (default: 15000)
    --odonymes        Path to odonymes.txt
"""

In [ ]:
import argparse
import os
import sys
import time

In [ ]:
parser = argparse.ArgumentParser()
parser.add_argument("input_file")
parser.add_argument("--gpu",            type=str, default="3")
parser.add_argument("--biaises-file",   default="./data/biaises_identified.csv")
parser.add_argument("--reference-file", default=None)
parser.add_argument("--sample-size",    type=int, default=15000)
parser.add_argument("--odonymes",       default="./data/odonymes.txt")
args = parser.parse_args()

In [ ]:
os.environ["CUDA_VISIBLE_DEVICES"] = args.gpu

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from datetime import datetime
import cudf

In [ ]:
BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
sys.path.insert(0, BASE_DIR)
from methods.methods import find_pos_elem
from methods.chunkcsv import process_dataframe_in_chunks2

In [ ]:
date = datetime.today().strftime("%Y%m%d")
print(f"[03] start — date={date}  gpu={args.gpu}")

In [ ]:
# Output directories — all created up-front with exist_ok
DIRS = {
    "biaised":   "./data/reference/ref_biaised/",
    "chunked":   "./data/reference/ref_chunked/",
    "geo":       "./data/reference/ref_chunked_geocoded/",
    "out":       "./data/reference/ref_biaised_geocoded/",
}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# §1  CUDF — Load patient data + compute bias positions
# ═════════════════════════════════════════════════════════════════════════════
print("[03] §1 Loading and position analysis (cudf)...")

In [ ]:
data = cudf.read_csv(args.input_file, sep=";").drop(
    columns=["Unnamed: 0"], errors="ignore"
)
df_odonyme = cudf.read_csv(args.odonymes, delimiter="|")
df_biaises = (
    cudf.read_csv(args.biaises_file, delimiter="|",
                  names=["index", "biais"], index_col="index")
    .drop(index=[None], errors="ignore")
)

In [ ]:
odonyme = df_odonyme["terme"].str.upper().drop_duplicates()
biais   = df_biaises["biais"].str.upper()

In [ ]:
print(f"  Patient rows: {len(data):,}  |  Biases: {len(df_biaises)}")

In [ ]:
df_pos = find_pos_elem(data, "adr_init", biais,
                       "biais", "contains_biais", "pos_biais",
                       drop_col_elem=False)

In [ ]:
# Transfer to pandas for the position comparison logic (row-level conditionals)
df = df_pos.to_pandas()
del data, df_pos        # free GPU memory

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# §2  PANDAS — Dominant bias position per bias token
# ═════════════════════════════════════════════════════════════════════════════
print("[03] §2 Deriving dominant bias positions (pandas)...")

In [ ]:
df["comp_biais_street"] = "NULL"

In [ ]:
# Addresses with street type + bias + residual unmatched token
mask_residual = (
    df["contains_street_type"] &
    df["contains_biais"] &
    df["not_matched_brute"].notna()
)
diff = df.loc[mask_residual, "pos_street_type"] - df.loc[mask_residual, "pos_biais"]
df.loc[mask_residual, "comp_biais_street"] = np.where(
    diff > 0, "BEFORE", np.where(diff < 0, "AFTER", "SAME_POS")
)

In [ ]:
# Fully matched (bias absorbed by geocoder — not a real bias signal)
mask_matched = (
    df["contains_street_type"] &
    df["contains_biais"] &
    df["not_matched_brute"].isna()
)
df.loc[mask_matched, "comp_biais_street"] = "MATCHED"

In [ ]:
# No street type
mask_no_street = (~df["contains_street_type"]) & df["contains_biais"]
df.loc[mask_no_street, "comp_biais_street"] = "NO STREET"

In [ ]:
# Only keep genuinely biased addresses for the pivot
df_biaised = df[df["contains_biais"] & (df["comp_biais_street"] != "MATCHED")]
print(f"  Biased addresses: {len(df_biaised):,}  ({100*len(df_biaised)/len(df):.2f}%)")

In [ ]:
counts = df_biaised.groupby(["biais", "comp_biais_street"]).size().reset_index(name="n")
totals = counts.groupby("biais")["n"].sum().rename("total")
counts = counts.join(totals, on="biais")
counts["pct"] = 100 * counts["n"] / counts["total"]

In [ ]:
pivot = counts.pivot(index="biais", columns="comp_biais_street", values="pct")
dict_pos_max = pivot.idxmax(axis=1).to_dict()

In [ ]:
print("  Dominant positions:")
for b, pos in dict_pos_max.items():
    print(f"    {b:15s} → {pos}")

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# §3  PANDAS — Load reference dataset and inject biases
# ═════════════════════════════════════════════════════════════════════════════
print("[03] §3 Loading reference dataset...")

In [ ]:
if args.reference_file:
    ref_path = args.reference_file
else:
    ref_dir = "./data/reference/"
    candidates = [
        f for f in os.listdir(ref_dir)
        if f.endswith(".csv") and "etablissements" in f
    ]
    if not candidates:
        raise FileNotFoundError(
            f"No reference CSV found in {ref_dir}. Pass --reference-file explicitly."
        )
    ref_path = os.path.join(ref_dir, candidates[0])
    print(f"  Auto-detected: {ref_path}")

In [ ]:
df_ref = pd.read_csv(ref_path, sep=";", dtype=str)

In [ ]:
# Metropolitan France only
mask_dom = (
    df_ref["code_postal_uai"].str.startswith("97") |
    df_ref["code_postal_uai"].str.startswith("98") |
    df_ref["code_postal_uai"].str.startswith("99")
)
df_ref_metrop = df_ref[~mask_dom].dropna(subset=["adresse_uai"]).copy()
df_ref_metrop["adresse_uai"] = df_ref_metrop["adresse_uai"].str.upper()

In [ ]:
KEEP_COLS = [
    "numero_uai", "adresse_uai", "localite_acheminement_uai",
    "code_postal_uai", "coordonnee_x", "coordonnee_y", "latitude", "longitude",
]
df_sample = df_ref_metrop.sample(args.sample_size)[KEEP_COLS]
print(f"  Reference sample: {len(df_sample):,} rows")

In [ ]:
print("[03] §3b Generating biased reference files...")

In [ ]:
def inject_bias(df_src: pd.DataFrame, bias: str, pos: str, col: str) -> pd.DataFrame:
    out = df_src.copy()
    if pos == "AFTER":
        out[col] = out[col] + f" {bias}"
    elif pos == "BEFORE":
        out[col] = f"{bias} " + out[col]
    return out

In [ ]:
for bias_token, position in dict_pos_max.items():
    out_path = os.path.join(DIRS["biaised"], f"df_ref_{bias_token}.csv")
    inject_bias(df_sample, bias_token, position, "adresse_uai").to_csv(out_path)
    print(f"  Saved: {out_path}")

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# §4  PANDAS — Geocode biased datasets in chunks (CPU/HTTP-bound)
# ═════════════════════════════════════════════════════════════════════════════
print("[03] §4 Geocoding biased datasets...")

In [ ]:
GEOCODE_COLS = ["adresse_uai", "code_postal_uai", "localite_acheminement_uai"]
OUT_COLS = [
    "numero_uai", "adresse_uai", "localite_acheminement_uai", "code_postal_uai",
    "x_L93_ref", "y_L93_ref", "x_WGS84_ref", "y_WGS84_ref",
    "latitude", "longitude",
    "result_housenumber", "result_name", "result_postcode",
    "result_city", "result_label", "label",
]

In [ ]:
t0 = time.time()
bias_files = [
    f for f in os.listdir(DIRS["biaised"])
    if os.path.isfile(os.path.join(DIRS["biaised"], f))
]

In [ ]:
for fname in tqdm(bias_files):
    label = fname.split("_")[2].split(".")[0]

    df_to_geocode = (
        pd.read_csv(os.path.join(DIRS["biaised"], fname), index_col=0)
        .rename(columns={
            "coordonnee_x": "x_L93_ref",
            "coordonnee_y": "y_L93_ref",
            "longitude":    "x_WGS84_ref",
            "latitude":     "y_WGS84_ref",
        })
    )
    df_to_geocode["label"] = label

    process_dataframe_in_chunks2(
        df=df_to_geocode,
        label=label,
        chunk_size=1000,
        chunks_dir_path=DIRS["chunked"],
        chunks_geo_dir_path=DIRS["geo"],
        columns={"columns": GEOCODE_COLS},
    )

    geocoded_files = [
        f for f in os.listdir(DIRS["geo"])
        if os.path.isfile(os.path.join(DIRS["geo"], f)) and f.startswith(f"{label}_")
    ]
    if not geocoded_files:
        print(f"  Warning: no geocoded chunks for '{label}' — skipping.")
        continue

    result = pd.concat(
        [pd.read_csv(os.path.join(DIRS["geo"], f)) for f in geocoded_files],
        ignore_index=True,
    )
    result = result[[c for c in OUT_COLS if c in result.columns]]
    result.to_csv(os.path.join(DIRS["out"], f"{label}_ref_biaised.csv"), sep=";")
    print(f"  Geocoded → {DIRS['out']}{label}_ref_biaised.csv")

In [ ]:
elapsed = (time.time() - t0) / 60
print(f"[03] DONE in {elapsed:.1f} min.")